### Libraries

In [5]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd



#### Load, Clean and Merge Bike Sensors Data with Sites and Directions

In [2]:
#Load Sites
site_columns = ["sensor_id", "site_nr", "longitude", "latitude", "name",
                "domain", "road_number", "district", "municipality", "interval", "installation_date"]
sites = pd.read_csv("../data/raw/sites.csv", header=None, names=site_columns)
print(f"Sites: {len(sites):,} rows")

#Load directions
direction_columns = ["sensor_id", "direction", "direction_name"]
directions = pd.read_csv("../data/raw/richtingen.csv", header=None, names=direction_columns)
print(f"Directions: {len(directions):,} rows")


Sites: 151 rows
Directions: 305 rows


In [4]:
#Load Bike Sensors
folder = Path("../data/raw/sensors")
files = list(folder.glob("data-*.csv"))

column_names = ["sensor_id", "direction", "vehicle_type", "start_time", "end_time", "count"]

dfs = []
for file in files:
    df = pd.read_csv(file, header=None, names=column_names)
    
    # Filter cyclists only
    df = df[df["vehicle_type"] == "FIETSERS"]

    #Drop problematic sensors
    df = df[~df["sensor_id"].isin([144, 123])]

    #Drop nulls
    df = df.dropna(subset=["count"])

    #Convert to datatime 
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["hour"] = df["start_time"].dt.floor("h")

    #Group by hor
    df = df.groupby(["sensor_id", "direction", "hour"], as_index=False)["count"].sum()


    #Merge with sites
    df = df.merge(sites, on="sensor_id", how="left")

    #Merge with directions
    df = df.merge(directions, on=["sensor_id", "direction"], how="left")
    
    dfs.append(df)

#Concatenate data
bike_data = pd.concat(dfs, ignore_index=True)


print(f"Total rows: {len(bike_data):,}")
print(f"Total columns: {bike_data.shape[1]}")





Total rows: 6,994,453
Total columns: 15


#### Merge Bike Sensor Data with Sites and Directions

#### Load and Clean Accidents Data

In [7]:
accidents = pd.read_excel("../data/raw/OPENDATA_MAP_2017-2024.xlsx")

# Filter Flanders + bikes
accidents = accidents[accidents["TX_RGN_COLLISION_NL"] == "Vlaams Gewest"]
accidents = accidents[
    (accidents["TX_ROAD_USR_TYPE1_NL"] == "Fiets") |
    (accidents["TX_ROAD_USR_TYPE2_NL"] == "Fiets")
]
print(f"After Flanders + bike filter: {len(accidents):,} rows")

#Filter year>= 2020
accidents = accidents[accidents["DT_YEAR_COLLISION"] >= 2020]
print(f"After year filter (2020-2024): {len(accidents):,} rows")

# Drop unknown hour
accidents = accidents[accidents["DT_TIME"] != 99]
print(f"After dropping unknown hour: {len(accidents):,} rows")

# Drop missing coordinates
accidents = accidents.dropna(subset=["MS_X_COORD", "MS_Y_COORD"])
print(f"After dropping missing coordinates: {len(accidents):,} rows")

# Keep only relevant columns
columns_to_keep = [
    "DT_YEAR_COLLISION", "DT_MONTH_COLLISION", "DT_TIME",
    "MS_X_COORD", "MS_Y_COORD", "TX_MUNTY_COLLISION_NL",
    "TX_CROSSWAY_NL", "CD_ROAD_TYPE_NL", "TX_BUILD_UP_AREA_NL",
    "TX_WEATHER_NL", "TX_ROAD_CONDITION_NL", "TX_LIGHT_CONDITION_NL",
    "TX_CLASS_ACCIDENTS_NL", "TX_COLLISION_TYPE_NL",
    "TX_ROAD_USR_TYPE1_NL", "TX_ROAD_USR_TYPE2_NL", "TX_OBSTACLES_NL"
]
accidents = accidents[columns_to_keep]
print(f"Columns kept: {len(accidents.columns)}")

After Flanders + bike filter: 59,683 rows
After year filter (2020-2024): 38,275 rows
After dropping unknown hour: 38,275 rows
After dropping missing coordinates: 35,195 rows
Columns kept: 17


#### Spatial Join (Accidents + Sensors)

In [8]:

# ── SPATIAL JOIN ACCIDENTS ↔ SENSORS ──────────────────────────────────────────

# 1. Get unique sensor locations
unique_sensors = bike_data.drop_duplicates(subset=["sensor_id"])[
    ["sensor_id", "longitude", "latitude", "name", "municipality"]
]

# 2. Convert accidents to GeoDataFrame (Belgian coordinate system)
accidents_gdf = gpd.GeoDataFrame(
    accidents,
    geometry=gpd.points_from_xy(accidents["MS_X_COORD"], accidents["MS_Y_COORD"]),
    crs="EPSG:31370"
)

# 3. Convert sensors to GeoDataFrame and reproject to match accidents
sensors_gdf = gpd.GeoDataFrame(
    unique_sensors,
    geometry=gpd.points_from_xy(unique_sensors["longitude"], unique_sensors["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:31370")

# 4. Find nearest sensor for each accident (max 500m)
accidents_with_sensor = gpd.sjoin_nearest(
    accidents_gdf,
    sensors_gdf,
    how="left",
    distance_col="distance_to_sensor"
)

accidents_with_sensor = accidents_with_sensor[
    accidents_with_sensor["distance_to_sensor"] <= 500
]

print(f"✅ Accidents within 500m of a sensor: {len(accidents_with_sensor):,}")
print(f"   ({len(accidents_with_sensor)/len(accidents)*100:.1f}% of cleaned accidents)")

✅ Accidents within 500m of a sensor: 818
   (2.3% of cleaned accidents)
